In [160]:
import os
os.environ["KERAS_BACKEND"]="torch"
import torch
import gc
import json
import collections
from pathlib import Path
from tqdm.auto import tqdm
import shutil
import numpy as np
from PIL import Image
import tensorflow as tf
import keras
from keras import layers
from keras.applications.inception_v3 import InceptionV3
from sklearn.utils import shuffle

# Chemins et imports
Le notebook sélectionne le backend Torch avant d’importer Keras. Il cherche le dossier models à partir du répertoire de travail, puis définit les chemins des modèles et des données.

In [ ]:
PROJECT_DIR = os.path.abspath(os.getcwd())
MODELS_DIR = os.path.join(PROJECT_DIR, "models")
CLASSIFIER_WEIGHTS = os.path.join(MODELS_DIR, "model_full.weights.h5")
DENOISER_MODEL = os.path.join(MODELS_DIR, "unet_denoiser_livrable2.keras")
DATASETS_DIR = os.path.join(PROJECT_DIR, "Leyenda", "datasets")
COCO_DIR = os.path.join(DATASETS_DIR, "train2014")
PHOTOS_DIR = os.path.join(DATASETS_DIR, "photos")
DENOISED_DIR = os.path.join(DATASETS_DIR, "denoised_images")
ANNOTATIONS_DIR = os.path.join(DATASETS_DIR, "annotations")
ANNOTATION_FILE = os.path.join(ANNOTATIONS_DIR, "captions_train2014.json")

# Etape 1 : classification Photo / Non-photo
Le modèle du livrable 1 reconnaît cinq classes : Painting, Photo, Schematics, Sketch et Text. Il faut donc garder ses cinq sorties : l’étape est dite binaire parce qu’on transforme ensuite sa prédiction en deux groupes, Photo et Non-photo.

Pour chaque image, le code :

la redimensionne en 128 × 128 ;
la passe au modèle, qui divise ses pixels par 255 grâce à sa couche Rescaling ;
récupère la probabilité associée à Photo ;
la compare au seuil 0,2301, défini dans le livrable 1.
Si la probabilité atteint le seuil, l’image est considérée comme une photo. Sinon, elle reste dans son dossier d’origine. Le code déplace les photos retenues dans datasets/photos — il ne les copie pas — mais ne déplace pas et ne supprime pas les autres images.

In [162]:
CLASS_NAMES = ["Painting", "Photo", "Schematics", "Sketch", "Text"]
NUM_CLASSES = len(CLASS_NAMES)
PHOTO_CLASS_INDEX = CLASS_NAMES.index("Photo")
THRESHOLD = 0.2301

In [163]:
def load_classifier_model():
    model = keras.models.load_model(CLASSIFIER_WEIGHTS, compile=False)
    return model

def load_image_as_array(path, target_size=(128, 128)):
    with Image.open(path) as image:
        image = image.convert("RGB").resize(target_size)
        return np.asarray(image, dtype=np.float32)

def classify_images(image_dir: str | Path,classifier: keras.models.Model):
    results = []
    image_dir = Path(image_dir)

    for image_file in tqdm(image_dir.iterdir()):
        image_array = load_image_as_array(str(image_file))
        image_array = np.expand_dims(image_array, axis=0)

        probabilities = classifier.predict(image_array,verbose=0)
        proba_photo = float(probabilities[0][PHOTO_CLASS_INDEX])

        results.append((str(image_file),proba_photo,proba_photo >= THRESHOLD))

    return results

def move_photos_to_folder(classification_results, dest_dir=PHOTOS_DIR):
    os.makedirs(dest_dir, exist_ok=True)
    moved = []
    for path, _, is_photo in classification_results:
        if is_photo:
            if os.path.abspath(os.path.dirname(path)) != os.path.abspath(dest_dir):
                filename = os.path.basename(path)
                dest_path = os.path.join(dest_dir, filename)
                if os.path.exists(dest_path):
                    base, extension = os.path.splitext(filename)
                    suffix = 1
                    while os.path.exists(dest_path):
                        dest_path = os.path.join(dest_dir, f"{base}_{suffix}{extension}")
                        suffix += 1
                shutil.move(path, dest_path)
                path = dest_path
            moved.append(path)
    return moved

# Etape 2 : débruitage avec le U-Net du livrable 2
Chaque photo retenue est chargée en RGB, redimensionnée en 299 × 299 et convertie en valeurs entre 0 et 1. Le U-Net du livrable 2 est ensuite appliqué à l’image, et le résultat est sauvegardé dans datasets/denoised_images.

Le U-Net effectue des opérations de pooling qui nécessitent des dimensions compatibles avec des réductions par deux. Le code complète donc temporairement les dimensions de 299 à 300, puis recadre le résultat à 299 × 299 avant de le sauvegarder.

In [164]:
def load_denoiser_model():
    denoiser = keras.models.load_model(DENOISER_MODEL, compile=False)
    return denoiser

def load_image_for_denoiser(path, target_size=(299, 299)):
    with Image.open(path) as image:
        image = image.convert("RGB").resize(target_size)
        return np.asarray(image, dtype=np.float32) / 255.0

def denoise_images(image_paths, denoiser, output_dir=DENOISED_DIR):
    os.makedirs(output_dir, exist_ok=True)

    for path in tqdm(image_paths):
        arr = load_image_for_denoiser(path)
        height, width = arr.shape[:2]
        pad_height = (-height) % 4
        pad_width = (-width) % 4
        padded = np.pad(arr, ((0, pad_height), (0, pad_width), (0, 0)), mode="edge")
        batch = np.expand_dims(padded, axis=0)
        denoised = keras.ops.convert_to_numpy(denoiser(batch, training=False))[0]
        denoised = denoised[:height, :width, :]
        denoised = np.clip(denoised, 0.0, 1.0)

        denoised_img = Image.fromarray(np.rint(denoised * 255).astype("uint8"))
        filename = os.path.basename(path)
        out_path = os.path.join(output_dir, filename)
        denoised_img.save(out_path)
        gc.collect()
        torch.cuda.empty_cache()


# Etape 3: Chargement des données

## Création des vecteurs captions et images

In [ ]:
def create_captions_and_path_vectors(annotation_file):
    image_path_to_caption = collections.defaultdict(list)
    train_captions = []
    img_name_vector = []
    mark_start, mark_end = '<start> ', ' <end>'

    with open(annotation_file, 'r') as f:
        annotations = json.load(f)

    for val in annotations['annotations']:
        caption =  mark_start + val['caption'].strip() + mark_end
        image_path = os.path.join(DENOISED_DIR,'COCO_train2014_' + '%012d.jpg' % (val['image_id']) +'.npy')

        if not os.path.isfile(image_path):
            continue
        image_path_to_caption[image_path].append(caption)

    train_image_paths = list(image_path_to_caption.keys())


    for image_path in train_image_paths:
        caption_list = image_path_to_caption[image_path]
        train_captions.extend(caption_list)
        [img_name_vector.append(image_path) for _ in range(len(caption_list))]

    return img_name_vector, train_captions

## Prétraitement des annotations

In [166]:
def calc_max_length(tensor):
    return max(len(t) for t in tensor)

def annotation_preprocessing(train_captions, top_k=5000):
    tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=top_k,
                                                    oov_token="<unk>",
                                                    filters='!"#$%&()*+.,-/:;=?@[\\]^_`{|}~')

    tokenizer.fit_on_texts(train_captions)
    tokenizer.word_index['<pad>'] = 0
    tokenizer.index_word[0] = '<pad>'
    train_seqs = tokenizer.texts_to_sequences(train_captions)
    cap_vector = keras.preprocessing.sequence.pad_sequences(train_seqs, padding='post')
    max_length = calc_max_length(train_seqs)

    return cap_vector, max_length, tokenizer

# Etape 4: Extraction des caractéristiques avec InceptionV3

In [167]:
def build_inception_feature_extractor():
    # Charge InceptionV3 pré-entraîné sur ImageNet, sans la tête de classification
    image_model = InceptionV3(weights="imagenet", include_top=False)
    new_input = keras.Input(shape = (299, 299, 3))
    hidden_layer = image_model(new_input)

    return keras.Model(new_input, hidden_layer)

def load_image_for_inception(path, target_size=(299, 299)):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, target_size)
    img = (img / 127.5) - 1.0

    return img, path

def extract_and_save_inception_features(img_name_vector, feature_extractor, batch_size=8):
    encode_train = sorted(set(img_name_vector))

    image_dataset = tf.data.Dataset.from_tensor_slices(encode_train)
    image_dataset = image_dataset.map(
    load_image_for_inception, num_parallel_calls=1).batch(batch_size)

    with torch.inference_mode():
        for img, path in tqdm(image_dataset):
            batch_features = feature_extractor(img, training=False)
            batch_features = keras.ops.reshape(batch_features,
                                        (batch_features.shape[0], -1, batch_features.shape[3]))
            for bf, p in zip(batch_features, path):
                bactch_feature = keras.ops.convert_to_numpy(bf)
                path_of_feature = p.numpy().decode("utf-8")
                np.save(path_of_feature, bactch_feature)

            del img
            del path
            del batch_features

            if torch.cuda.is_available():
                torch.cuda.empty_cache()


# Etape 5 : séparation train / validation (80/20)

In [ ]:
def map_func(img_name, cap):
    img_tensor = np.load(img_name.decode('utf-8'))
    return img_tensor, cap

def data_splitting(img_name_vector, cap_vector, buffer_size, batch_size):

    img_to_cap_vector = collections.defaultdict(list)
    for img, cap in zip(img_name_vector, cap_vector):
        img_to_cap_vector[img].append(cap)

    img_keys = list(img_to_cap_vector.keys())
    img_keys_shuffled = shuffle(img_keys)
    slice_index = int(len(img_keys)*0.8)
    img_name_train_keys, img_name_val_keys = img_keys_shuffled[:slice_index], img_keys_shuffled[slice_index:]

    # boucle pour le jeu d'entrainement
    img_name_train = []
    cap_train = []
    for imgt in img_name_train_keys:
        capt_len = len(img_to_cap_vector[imgt])
        img_name_train.extend([imgt] * capt_len)
        cap_train.extend(img_to_cap_vector[imgt])

    # Boucle pour le jeu de test
    img_name_val = []
    cap_val = []
    for imgv in img_name_val_keys:
        capv_len = len(img_to_cap_vector[imgv])
        img_name_val.extend([imgv] * capv_len)
        cap_val.extend(img_to_cap_vector[imgv])

    dataset = tf.data.Dataset.from_tensor_slices((img_name_train, cap_train))
    dataset = dataset.map(lambda item1, item2: tf.numpy_function(
            map_func, [item1, item2], [tf.float32, tf.int32]),
            num_parallel_calls=tf.data.experimental.AUTOTUNE)

    # Melanger les donnees et les diviser en batchs
    dataset = dataset.shuffle(buffer_size).batch(batch_size)
    dataset = dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

    return dataset

# Etape 5 : pipeline complète

In [169]:
# def run_pipeline():
#     print(f"1. Classification des images")
#     classifier = load_classifier_model()
#     classification_results = classify_images(COCO_DIR, classifier)
#     photos_moved = move_photos_to_folder(classification_results)
#     print(f"{len(photos_moved)} photos deplacees dans {PHOTOS_DIR}.")
#     print("2. Débruitage")
#     print("3. Chargement des donnees et pretraitement")
#     img_name_vector, train_captions = create_captions_and_path_vectors(annotation_file=ANNOTATION_FILE)
#     cap_vector, max_length, tokenizer = annotation_preprocessing(train_captions)

#     print("4. Extraction des caracteristiques avec InceptionV3")
#     feature_extractor = build_inception_feature_extractor()
#     extract_and_save_inception_features(img_name_vector, feature_extractor)
#     print(f"Extraction terminee dans : {DENOISED_DIR}")

#     print("5. Séparation train/validation/test")
#     dataset = data_splitting(img_name_vector, cap_vector, buffer_size=1000, batch_size=64)

#     return dataset, cap_vector, max_length, train_captions


In [170]:
# Étape 7 : Modèle de captioning (Encoder–Attention–Decoder)


À partir d'ici, on construit le pipeline de génération de légendes. L'architecture suit le schéma classique **Show, Attend and Tell** :

1. **CNN Encoder** — projette les feature maps spatiales d'InceptionV3 dans l'espace d'embedding.
2. **Bahdanau Attention** — apprend à pondérer les régions de l'image en fonction de l'état caché du décodeur.
3. **RNN Decoder (LSTM)** — génère la légende mot par mot en utilisant le contexte visuel et l'état caché.

**Prérequis** : les images ont déjà été traitées par votre collègue (classification, débruitage). On part du principe que vous disposez d'un dossier d'images prêtes à l'emploi et d'un fichier de captions associé.

In [171]:
import time
import json
import matplotlib.pyplot as plt
import keras
from keras import layers, ops
import numpy as np
from PIL import Image
from keras.applications.inception_v3 import InceptionV3, preprocess_input

## 7.1 – Hyperparamètres

| Paramètre | Valeur | Rôle |
|---|---|---|
| `EMBEDDING_DIM` | 256 | Taille du vecteur d'embedding pour les mots et les features visuelles |
| `UNITS` | 512 | Nombre d'unités LSTM (taille de l'état caché) |
| `VOCAB_SIZE` | à définir | Taille du vocabulaire (nombre de mots uniques + tokens spéciaux) |
| `MAX_LENGTH` | à définir | Longueur maximale d'une légende (en tokens) |

**Note** : `VOCAB_SIZE` et `MAX_LENGTH` seront définis après le chargement et la tokenisation des captions par votre collègue. Vous devrez les adapter en fonction du vocabulaire réel.

In [172]:
# ── Hyperparamètres du modèle ──
EMBEDDING_DIM = 256
UNITS = 512
BATCH_SIZE = 64
EPOCHS = 20
LEARNING_RATE = 1e-4

# ── À adapter selon votre vocabulaire ──
# VOCAB_SIZE = len(tokenizer.word_index) + 1
# MAX_LENGTH = max(len(cap) for cap in all_captions)
# Pour le moment, on met des valeurs par défaut :
VOCAB_SIZE = 5001     # à remplacer par la vraie taille du vocabulaire
MAX_LENGTH = 40       # à remplacer par la vraie longueur max

print(f"Embedding dim : {EMBEDDING_DIM}")
print(f"LSTM units    : {UNITS}")
print(f"Vocab size    : {VOCAB_SIZE}")
print(f"Max length    : {MAX_LENGTH}")

Embedding dim : 256
LSTM units    : 512
Vocab size    : 5001
Max length    : 40


## 7.2 – Extracteur de features spatiales (InceptionV3)

Contrairement à l'étape 3 qui utilisait InceptionV3 avec `pooling='avg'` (un seul vecteur de 2048 par image), ici on a besoin des **feature maps spatiales** pour que l'attention puisse se focaliser sur différentes régions de l'image.

On utilise la sortie de la dernière couche convolutive d'InceptionV3 (`mixed7`), qui produit un tensor de forme `(8, 8, 2048)` — soit 64 régions de 2048 dimensions chacune. Ce tensor est ensuite aplati en `(64, 2048)` avant d'être passé à l'encodeur CNN.

In [173]:
def create_captions_and_path_vectors(annotation_file):
    image_path_to_caption = collections.defaultdict(list)
    train_captions = []
    img_name_vector = []
    mark_start, mark_end = '<start> ', ' <end>'

    with open(annotation_file, 'r') as f:
        annotations = json.load(f)

    for val in annotations['annotations']:
        caption =  mark_start + val['caption'].strip() + mark_end
        image_path = os.path.join(DENOISED_DIR,'COCO_train2014_' + '%012d.jpg' % (val['image_id']))

        if not os.path.isfile(image_path):
            continue
        image_path_to_caption[image_path].append(caption)

    train_image_paths = list(image_path_to_caption.keys())


    for image_path in train_image_paths:
        caption_list = image_path_to_caption[image_path]
        train_captions.extend(caption_list)
        [img_name_vector.append(image_path) for _ in range(len(caption_list))]

    return img_name_vector, train_captions

In [174]:
def map_func(img_name, cap):
    img_tensor = np.load(img_name.decode('utf-8')+'.npy')
    return img_tensor, cap

def data_splitting(img_name_vector, cap_vector, buffer_size, batch_size):

    img_to_cap_vector = collections.defaultdict(list)
    for img, cap in zip(img_name_vector, cap_vector):
        img_to_cap_vector[img].append(cap)

    img_keys = list(img_to_cap_vector.keys())
    img_keys_shuffled = shuffle(img_keys)
    slice_index = int(len(img_keys)*0.8)
    img_name_train_keys, img_name_val_keys = img_keys_shuffled[:slice_index], img_keys_shuffled[slice_index:]

    # boucle pour le jeu d'entrainement
    img_name_train = []
    cap_train = []
    for imgt in img_name_train_keys:
        capt_len = len(img_to_cap_vector[imgt])
        img_name_train.extend([imgt] * capt_len)
        cap_train.extend(img_to_cap_vector[imgt])

    # Boucle pour le jeu de test
    img_name_val = []
    cap_val = []
    for imgv in img_name_val_keys:
        capv_len = len(img_to_cap_vector[imgv])
        img_name_val.extend([imgv] * capv_len)
        cap_val.extend(img_to_cap_vector[imgv])

    dataset = tf.data.Dataset.from_tensor_slices((img_name_train, cap_train))
    dataset = dataset.map(lambda item1, item2: tf.numpy_function(
            map_func, [item1, item2], [tf.float32, tf.int32]),
            num_parallel_calls=tf.data.experimental.AUTOTUNE)

    # Melanger les donnees et les diviser en batchs
    dataset = dataset.shuffle(buffer_size).batch(batch_size)
    dataset = dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

    return dataset

## 7.3 – CNN Encoder

L'encodeur CNN reçoit les features spatiales d'InceptionV3 de forme `(batch, 64, 2048)` et les projette dans l'espace d'embedding via une couche Dense suivie d'une activation ReLU.

**Entrée** : `(batch_size, 64, 2048)` — 64 régions spatiales, chacune de 2048 dimensions
**Sortie** : `(batch_size, 64, embedding_dim)` — les mêmes 64 régions projetées dans l'espace d'embedding

In [175]:
class CNN_Encoder(keras.Model):
    """
    Encodeur CNN : projette les features spatiales InceptionV3
    dans l'espace d'embedding.

    Entrée  : (batch, 64, 2048)
    Sortie  : (batch, 64, embedding_dim)
    """

    def __init__(self, embedding_dim):
        super(CNN_Encoder, self).__init__()
        # Couche fully connected appliquée à chaque région spatiale
        self.fc = layers.Dense(embedding_dim)

    def call(self, x):
        # x : (batch, 64, 2048)
        x = self.fc(x)        # -> (batch, 64, embedding_dim)
        x = ops.relu(x)       # activation ReLU
        return x

## 7.4 – Mécanisme d'attention de Bahdanau

L'attention de Bahdanau (attention additive) permet au décodeur de « regarder » différentes régions de l'image à chaque pas de temps. Le mécanisme calcule un score pour chaque région, puis normalise ces scores via softmax pour obtenir des poids d'attention.

**Formule** :
```
score(i) = V · tanh(W_features · features(i) + W_hidden · hidden)
attention_weights = softmax(scores)
context = Σ attention_weights(i) × features(i)
```

**Entrées** :
- `features` : `(batch, 64, embedding_dim)` — les features encodées de l'image
- `hidden` : `(batch, units)` — l'état caché précédent du LSTM

**Sorties** :
- `context_vector` : `(batch, embedding_dim)` — le résumé visuel pondéré
- `attention_weights` : `(batch, 64, 1)` — les poids d'attention par région

In [176]:
class BahdanauAttention(layers.Layer):
    """
    Attention additive de Bahdanau.
    Calcule un vecteur de contexte en pondérant les features spatiales
    selon l'état caché courant du LSTM.
    """

    def __init__(self, units):
        super().__init__()
        self.W_features = layers.Dense(units)  # projette les features
        self.W_hidden = layers.Dense(units)     # projette l'état caché
        self.V = layers.Dense(1)                # score scalaire

    def call(self, features, hidden):
        # features : (batch, 64, embedding_dim)
        # hidden   : (batch, units)

        # Ajouter une dimension pour le broadcast : (batch, 1, units)
        hidden_expanded = ops.expand_dims(hidden, axis=1)

        # Calcul des scores : (batch, 64, 1)
        scores = self.V(
            ops.nn.tanh(
                self.W_features(features) + self.W_hidden(hidden_expanded)
            )
        )

        # Normalisation softmax sur l'axe des régions : (batch, 64, 1)
        attention_weights = ops.nn.softmax(scores, axis=1)

        # Contexte visuel pondéré : (batch, embedding_dim)
        context_vector = ops.sum(attention_weights * features, axis=1)

        return context_vector, attention_weights

## 7.5 – Décodeur LSTM avec attention

Le décodeur génère la légende mot par mot. À chaque pas de temps, il :

1. Calcule l'attention sur les features visuelles à partir de l'état caché précédent
2. Transforme le token courant en embedding
3. Concatène le contexte visuel et l'embedding du mot
4. Passe le tout dans une cellule LSTM
5. Prédit le prochain mot via deux couches Dense

**Pourquoi un LSTM plutôt qu'un GRU ?** Le LSTM possède deux états (hidden state `h` et cell state `c`) qui lui permettent de mieux gérer les dépendances longues dans les séquences. Cela est important pour les légendes qui peuvent être longues.

**Important** : le LSTM retourne 3 valeurs `(output, state_h, state_c)` alors qu'un GRU n'en retourne que 2. Il faut donc passer **et recevoir** les deux états à chaque pas de temps.

In [177]:
class RNN_Decoder(keras.Model):
    """
    Décodeur LSTM avec mécanisme d'attention de Bahdanau.

    À chaque pas de temps :
      1. Calcul de l'attention sur les features visuelles
      2. Embedding du token courant
      3. Concaténation contexte + embedding
      4. Passage dans le LSTM
      5. Prédiction du prochain token
    """

    def __init__(self, embedding_dim, units, vocab_size):
        super(RNN_Decoder, self).__init__()
        self.units = units

        # 1. Embedding des tokens
        self.embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim
        )

        # 2. Mécanisme d'attention
        self.attention = BahdanauAttention(self.units)

        # 3. Cellule LSTM
        self.lstm = layers.LSTM(
            units=self.units,
            return_sequences=True,
            return_state=True,
            recurrent_initializer='glorot_uniform'
        )

        # 4. Première couche dense (transformation non-linéaire)
        self.fc1 = layers.Dense(self.units)

        # 5. Couche de sortie : score pour chaque mot du vocabulaire
        self.fc2 = layers.Dense(vocab_size)

    def call(self, x, features, hidden, cell_state):
        """
        Paramètres :
            x          : (batch_size, 1)            — indice du token courant
            features   : (batch_size, 64, emb_dim)  — features encodées
            hidden     : (batch_size, units)        — état caché LSTM (h)
            cell_state : (batch_size, units)        — état cellule LSTM (c)

        Retourne :
            predictions       : (batch_size, vocab_size)
            state_h           : (batch_size, units)
            state_c           : (batch_size, units)
            attention_weights : (batch_size, 64, 1)
        """
        # 1. Attention : pondération des régions visuelles
        context_vector, attention_weights = self.attention(features, hidden)
        # context_vector : (batch, embedding_dim)

        # 2. Embedding du token courant
        x = self.embedding(x)
        # x : (batch, 1, embedding_dim)

        # 3. Ajouter la dimension temporelle au contexte
        context_vector = ops.expand_dims(context_vector, axis=1)
        # context_vector : (batch, 1, embedding_dim)

        # 4. Concaténation contexte + embedding
        x = ops.concatenate([context_vector, x], axis=-1)
        # x : (batch, 1, embedding_dim * 2)

        # 5. Passage dans le LSTM
        output, state_h, state_c = self.lstm(
            x,
            initial_state=[hidden, cell_state]
        )
        # output  : (batch, 1, units)
        # state_h : (batch, units)
        # state_c : (batch, units)

        # 6. Couches Dense pour la prédiction
        y = self.fc1(output)                    # (batch, 1, units)
        y = self.fc2(y)                         # (batch, 1, vocab_size)
        y = ops.squeeze(y, axis=1)              # (batch, vocab_size)

        return y, state_h, state_c, attention_weights

    def reset_state(self, batch_size):
        """
        Initialise les deux états du LSTM à zéro.
        Retourne (hidden, cell_state).
        """
        hidden     = ops.zeros((batch_size, self.units))
        cell_state = ops.zeros((batch_size, self.units))
        return hidden, cell_state

## 7.6 – Instanciation des modèles

On crée l'encodeur et le décodeur avec les hyperparamètres définis plus haut. L'optimiseur Adam est utilisé avec un learning rate faible (`1e-4`) pour un entraînement stable.

In [178]:
encoder = CNN_Encoder(EMBEDDING_DIM)
decoder = RNN_Decoder(
    embedding_dim=EMBEDDING_DIM,
    units=UNITS,
    vocab_size=VOCAB_SIZE
)

optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)

print("Encoder et Decoder instanciés.")

Encoder et Decoder instanciés.


## 7.7 – Fonction de perte

On utilise la **Sparse Categorical Crossentropy** avec `from_logits=True` (car la sortie du décodeur n'a pas de softmax). Les tokens de padding (indice 0) sont masqués pour ne pas pénaliser le modèle sur ces positions.

In [179]:
loss_object = keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none'
)


def loss_function(real, pred):
    """
    Calcule la perte en masquant les tokens de padding (indice 0).

    real : (batch_size,)      — indices des vrais tokens
    pred : (batch_size, vocab) — logits prédits par le décodeur
    """
    # Masque : True là où le token n'est PAS du padding
    mask = ops.cast(real != 0, dtype='float32')

    # Perte par token
    loss_ = loss_object(real, pred)

    # On masque les positions de padding
    loss_ = loss_ * mask

    return ops.mean(loss_)

## 7.8 – Boucle d'entraînement (train_step)

Chaque pas d'entraînement :

1. Encode les features visuelles avec `CNN_Encoder`
2. Initialise le décodeur avec le token `<start>`
3. À chaque pas de temps, prédit le prochain mot et utilise le **teacher forcing** (le vrai mot est donné en entrée à l'étape suivante)
4. Calcule la perte totale et met à jour les poids

**Note sur le LSTM** : contrairement à un GRU, on doit gérer **deux états** (`hidden` et `cell_state`). Le `train_step` prend bien en compte les 4 valeurs retournées par le décodeur : `(predictions, state_h, state_c, attention_weights)`.

In [180]:
def train_step(img_tensor, target, word_to_index):
    """
    Effectue un pas d'entraînement sur un batch.

    img_tensor    : (batch, 64, 2048) — features spatiales pré-extraites
    target        : (batch, max_length) — séquences de tokens (captions)
    word_to_index : dict — mapping mot -> indice (pour récupérer l'indice de <start>)
    """
    loss = 0
    batch_size_actual = ops.shape(target)[0]

    # Initialiser les deux états du LSTM à zéro
    hidden, cell_state = decoder.reset_state(batch_size_actual)

    # Token de départ : <start> pour chaque élément du batch
    start_token = word_to_index.get('<start>', 1)
    dec_input = ops.expand_dims(
        ops.full((batch_size_actual,), start_token, dtype='int32'),
        axis=1
    )

    # Encoder les features visuelles
    features = encoder(img_tensor)
    # features : (batch, 64, embedding_dim)

    # Teacher forcing : le vrai mot à l'étape t sert d'entrée à l'étape t+1
    seq_length = ops.shape(target)[1]
    for i in range(1, seq_length):
        # Prédiction avec le LSTM
        predictions, hidden, cell_state, _ = decoder(
            dec_input, features, hidden, cell_state
        )
        # predictions : (batch, vocab_size)

        # Accumulation de la perte
        loss += loss_function(target[:, i], predictions)

        # Teacher forcing : le vrai token de l'étape i
        dec_input = ops.expand_dims(target[:, i], axis=1)

    # Perte moyenne par pas de temps
    total_loss = loss / ops.cast(seq_length, dtype='float32')

    return loss, total_loss

## 7.9 – Sauvegarde des checkpoints

On sauvegarde les poids de l'encodeur et du décodeur toutes les 5 époques. Cela permet de reprendre l'entraînement en cas d'interruption.

In [181]:
CHECKPOINT_DIR = os.path.join(os.getcwd(), 'checkpoints', 'captioning')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


def save_checkpoint(encoder, decoder, optimizer, epoch, path=CHECKPOINT_DIR):
    """Sauvegarde manuelle des poids du modèle."""
    encoder.save_weights(os.path.join(path, f'encoder_epoch_{epoch}.weights.h5'))
    decoder.save_weights(os.path.join(path, f'decoder_epoch_{epoch}.weights.h5'))
    print(f'  Checkpoint sauvegardé (epoch {epoch})')


def load_checkpoint(encoder, decoder, epoch, path=CHECKPOINT_DIR):
    """Charge les poids d'un checkpoint donné."""
    encoder.load_weights(os.path.join(path, f'encoder_epoch_{epoch}.weights.h5'))
    decoder.load_weights(os.path.join(path, f'decoder_epoch_{epoch}.weights.h5'))
    print(f'  Checkpoint chargé (epoch {epoch})')


print(f'Dossier de checkpoints : {CHECKPOINT_DIR}')

Dossier de checkpoints : c:\Users\arist\Documents\CESI\BLoc Deep Learning\Deep_Learning_CNN\Livrable1\Leyenda\checkpoints\captioning


## 7.10 – Boucle d'entraînement complète

La boucle d'entraînement itère sur les époques et les batchs. À chaque batch, le `train_step` calcule la perte et met à jour les poids via la rétropropagation.

**À adapter avant exécution** :
- `dataset` : votre dataset (paires features/captions), par exemple un générateur Python
- `word_to_index` : le dictionnaire mot → indice issu de la tokenisation
- `num_steps` : le nombre total de batchs par époque

In [182]:
def create_dataset():
    img_name_vector, train_captions = create_captions_and_path_vectors(ANNOTATION_FILE)

    cap_vector, max_length, tokenizer = annotation_preprocessing(train_captions)
    
    dataset = data_splitting(img_name_vector, cap_vector, 1000, 32)

    return dataset, tokenizer, max_length


In [183]:
dataset, tokenizer, max_length = create_dataset()

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\arist\\Documents\\CESI\\BLoc Deep Learning\\Deep_Learning_CNN\\Livrable1\\Leyenda\\Leyenda\\datasets\\captions_train2014.json'

In [ ]:
for img_batch, cap_batch in dataset.take(1):
    print("Shape images   :", img_batch.shape)
    print("Shape légendes :", cap_batch.shape)

UnknownError: {{function_node __wrapped__IteratorGetNext_output_types_2_device_/job:localhost/replica:0/task:0/device:CPU:0}} Error in user-defined function passed to ParallelMapDatasetV2:1 transformation with iterator: Iterator::Root::Prefetch::FiniteTake::Prefetch::BatchV2::Shuffle::ParallelMapV2: FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\arist\\Documents\\CESI\\BLoc Deep Learning\\Deep_Learning_CNN\\Livrable1\\Leyenda\\Leyenda\\datasets\\denoised_images\\COCO_train2014_000000459670.jpg.npy.npy'
Traceback (most recent call last):

  File "c:\Users\arist\AppData\Local\Programs\Python\Python313\Lib\site-packages\tensorflow\python\ops\script_ops.py", line 269, in __call__
    ret = func(*args)

  File "c:\Users\arist\AppData\Local\Programs\Python\Python313\Lib\site-packages\tensorflow\python\autograph\impl\api.py", line 643, in wrapper
    return func(*args, **kwargs)

  File "C:\Users\arist\AppData\Local\Temp\ipykernel_10232\1603430996.py", line 2, in map_func

In [ ]:
# ══════════════════════════════════════════════════════════════
# IMPORTANT : Adaptez les variables ci-dessous à votre dataset
# ══════════════════════════════════════════════════════════════
dataset       =  # itérable de (img_tensor, target) batchés
word_to_index = tokenizer.word_to_index# ex: tokenizer.word_index
num_steps     = 32  # nombre de batchs par époque

loss_history = []

for epoch in range(EPOCHS):
    start_time = time.time()
    total_loss = 0

    for batch_idx, (img_tensor, target) in enumerate(dataset):
        batch_loss, t_loss = train_step(img_tensor, target, word_to_index)
        total_loss += t_loss

        if batch_idx % 100 == 0:
            print(f'  Epoch {epoch+1} | Batch {batch_idx} | '
               f'Loss: {batch_loss.numpy() / int(target.shape[1]):.4f}')

    avg_loss = total_loss / num_steps
    loss_history.append(float(avg_loss))

    # Sauvegarde périodique
    if (epoch + 1) % 5 == 0:
        save_checkpoint(encoder, decoder, optimizer, epoch + 1)

    elapsed = time.time() - start_time
    print(f'Epoch {epoch+1}/{EPOCHS} — Loss: {avg_loss:.6f} — '
        f'Durée: {elapsed:.1f}s')

# # ── Courbe d'entraînement ──
plt.figure(figsize=(10, 5))
plt.plot(loss_history, marker='o', linewidth=2)
plt.xlabel('Époque')
plt.ylabel('Perte moyenne')
plt.title('Courbe de perte — Entraînement du captioning')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Boucle d'entraînement prête. Décommentez le code ci-dessus pour lancer.")

SyntaxError: invalid syntax (4107446902.py, line 4)

## 7.11 – Évaluation : génération de légendes

La fonction `evaluate` prend une image en entrée et génère une légende mot par mot via un **décodage glouton** (greedy decoding) :

1. Extraire les features spatiales de l'image avec InceptionV3
2. Encoder les features
3. Décoder mot par mot jusqu'au token `<end>` ou jusqu'à `MAX_LENGTH`
4. Sauvegarder les poids d'attention pour la visualisation

On enregistre les poids d'attention à chaque pas pour pouvoir visualiser sur quelles régions le modèle se concentre pour chaque mot généré.

In [ ]:
def evaluate(image_path, encoder, decoder, spatial_extractor,
             word_to_index, index_to_word, max_length=MAX_LENGTH):
    """
    Génère une légende pour une image donnée.

    Retourne :
        result         : liste de mots prédits
        attention_plot : matrice (nb_mots, 64) des poids d'attention
    """
    attention_plot = np.zeros((max_length, 64))

    # 1. Charger et prétraiter l'image
    img_arr = load_and_preprocess_image(image_path)
    img_batch = np.expand_dims(img_arr, axis=0)  # (1, 299, 299, 3)

    # 2. Extraire les features spatiales : (1, 8, 8, 2048)
    spatial_feat = spatial_extractor(img_batch, training=False)
    spatial_feat = ops.convert_to_numpy(spatial_feat)
    # Reshape : (1, 64, 2048)
    spatial_feat = spatial_feat.reshape(1, -1, spatial_feat.shape[-1])

    # 3. Encoder
    features = encoder(spatial_feat)

    # 4. Initialiser le LSTM
    hidden, cell_state = decoder.reset_state(batch_size=1)

    # Token de départ
    start_token = word_to_index.get('<start>', 1)
    dec_input = ops.expand_dims(ops.array([start_token], dtype='int32'), axis=1)

    result = []

    for i in range(max_length):
        predictions, hidden, cell_state, attention_weights = decoder(
            dec_input, features, hidden, cell_state
        )

        # Sauvegarder l'attention : (1, 64, 1) -> (64,)
        att = ops.convert_to_numpy(attention_weights)
        attention_plot[i] = att.reshape(-1)

        # Sélection gloutonne (argmax)
        pred_np = ops.convert_to_numpy(predictions)
        predicted_id = int(np.argmax(pred_np[0]))

        # Convertir l'indice en mot
        predicted_word = index_to_word.get(predicted_id, '<unk>')
        result.append(predicted_word)

        # Arrêt si <end>
        if predicted_word == '<end>':
            break

        # Le mot prédit devient l'entrée suivante
        dec_input = ops.expand_dims(ops.array([predicted_id], dtype='int32'), axis=1)

    # Tronquer l'attention aux mots réellement générés
    attention_plot = attention_plot[:len(result), :]
    return result, attention_plot

## 7.12 – Visualisation de l'attention

La fonction `plot_attention` superpose les poids d'attention sur l'image pour chaque mot prédit. Cela permet de vérifier visuellement que le modèle « regarde » les bonnes régions de l'image pour chaque mot généré.

Chaque sous-figure montre :
- L'image originale en arrière-plan
- La carte d'attention (heatmap) en superposition
- Le mot prédit en titre

In [ ]:
def plot_attention(image_path, result, attention_plot):
    """
    Affiche l'image avec la carte d'attention pour chaque mot prédit.

    image_path     : chemin vers l'image originale
    result         : liste de mots prédits
    attention_plot : matrice (nb_mots, 64) des poids d'attention
    """
    temp_image = np.array(Image.open(image_path).convert('RGB'))

    n_words = len(result)
    n_cols = min(4, n_words)
    n_rows = (n_words + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    if n_words == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for idx in range(n_words):
        # Reshape (64,) -> (8, 8) pour correspondre à la grille spatiale
        temp_att = attention_plot[idx].reshape(8, 8)

        ax = axes[idx]
        ax.set_title(result[idx], fontsize=14, fontweight='bold')
        img_disp = ax.imshow(temp_image)
        ax.imshow(
            temp_att, cmap='jet', alpha=0.5,
            extent=img_disp.get_extent(),
            interpolation='bilinear'
        )
        ax.axis('off')

    # Masquer les axes inutilisés
    for idx in range(n_words, len(axes)):
        axes[idx].axis('off')

    plt.suptitle("Visualisation de l'attention par mot", fontsize=16)
    plt.tight_layout()
    plt.show()


# ── Exemple d'utilisation ──
# (décommentez après entraînement)
#
# spatial_extractor = build_spatial_feature_extractor()
# test_image = 'chemin/vers/votre/image.jpg'
# result, attention_plot = evaluate(
#     test_image, encoder, decoder, spatial_extractor,
#     word_to_index, index_to_word
# )
# print('Légende prédite :', ' '.join(result))
# plot_attention(test_image, result, attention_plot)